In [ ]:
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df = pd.read_csv('/content/House_Price_dataset.csv')
df = df[df['purpose'] == 'For Sale'].copy()
df['Area'] = df['Area Size']
df.loc[df['Area Type'] == 'Kanal', 'Area'] = df['Area Size'] * 20
df = df[(df['Area'] > 0) & (df['price'] > 0)]
df = df[(df['Area'] < df['Area'].quantile(0.99)) & (df['price'] < df['price'].quantile(0.99))]

X = df[['Area', 'price']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
print(df['Cluster'].value_counts().sort_index())
print(df.groupby('Cluster')[['Area', 'price']].mean())

plt.scatter(df['Area'], df['price'], c=df['Cluster'], cmap='viridis')
centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.xlabel('Area (Marla)')
plt.ylabel('Price')
plt.title('House Segments (K-Means, K=4)')
plt.legend()
plt.show()

# Note: Hierarchical clustering on large datasets can be extremely slow and memory-intensive. 
# We will use AgglomerativeClustering on a subset or run it directly if dataset size permits. 
# Since df has over 118k rows, we sample a subset of 5000 rows for Agglomerative Clustering to prevent running out of memory.
df_sample = df.sample(5000, random_state=42).copy()
X_sample_scaled = scaler.transform(df_sample[['Area', 'price']])
hc = AgglomerativeClustering(n_clusters=4)
df_sample['HC_Cluster'] = hc.fit_predict(X_sample_scaled)

plt.scatter(df_sample['Area'], df_sample['price'], c=df_sample['HC_Cluster'], cmap='plasma')
plt.xlabel('Area Size')
plt.ylabel('Price')
plt.title('House Segments (Hierarchical Clustering - Sample)')
plt.show()

## Preview
 
 ![Elbow method and K-Means segment graphs](experiment_10_preview.svg)
 
 *Illustrative preview. Running the notebook creates the dataset-based plots.*
